## Load All Phase 2 Results

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, avg, count, sum as spark_sum,
    round as spark_round, when, isnan
)
import sqlalchemy
import pandas as pd
import os

# ── Start SparkSession ──
spark = SparkSession.builder \
    .appName("LogiScale_Phase3_Visualization") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()                                   

# ── Create outputs folder ──
os.makedirs("outputs", exist_ok=True)

# ── Load cleaned data from Phase 1 ──
df = spark.read.csv(
    "dataco_cleaned.csv",
    header=True,
    inferSchema=True
)

print(f"Rows loaded: {df.count():,}")

# ── SQL engine ──
engine = sqlalchemy.create_engine("sqlite:///outputs/logiscale.db")
print("SQL engine connected: outputs/logiscale.db")

Rows loaded: 180,519
SQL engine connected: outputs/logiscale.db


##  Export All Aggregated Tables

In [2]:
print("\n" + "="*50)
print("  EXPORTING ALL TABLES → SQL + CSV")
print("="*50)

# Helper: saves one Spark DataFrame to both SQL and CSV
def export_table(spark_df, table_name):
    pdf = spark_df.toPandas()

    # Save to CSV
    csv_path = f"outputs/{table_name}.csv"
    pdf.to_csv(csv_path, index=False)

    # Save to SQL
    pdf.to_sql(table_name, engine, if_exists="replace", index=False)

    print(f"  [SAVED] {table_name:30s} → {len(pdf):>6,} rows")
    return pdf

# ── Rebuild all Phase 2 aggregations ──

# 1. Delay summary by route
from pyspark.sql.functions import stddev

df_route_delay = df.groupBy("route", "shipping_mode").agg(
    spark_round(avg("delay_days"),    2).alias("avg_delay_days"),
    spark_round(stddev("delay_days"), 2).alias("stddev_delay_days"),
    count("*").alias("total_orders"),
    spark_round(
        count(when(col("delivery_flag") == "Late",    1)) /
        count("*") * 100, 1
    ).alias("late_pct"),
    spark_round(
        count(when(col("delivery_flag") == "On Time", 1)) /
        count("*") * 100, 1
    ).alias("on_time_pct")
)

# 2. Warehouse summary
df_warehouse = df.groupBy("warehouse_id", "market").agg(
    count("*").alias("total_orders"),
    spark_round(spark_sum("units_sold"),      2).alias("total_units_sold"),
    spark_round(spark_sum("sales_amount"),    2).alias("total_sales"),
    spark_round(spark_sum("profit_per_order"),2).alias("total_profit"),
    spark_round(avg("delay_days"),            2).alias("avg_delay_days"),
    spark_round(
        count(when(col("late_risk") == 1, 1)) /
        count("*") * 100, 1
    ).alias("late_risk_pct")
)

# 3. Daily warehouse metrics (what the task card specifically asks for)
df_daily_warehouse = df.groupBy(
    "warehouse_id", "market", "order_date"
).agg(
    count("*").alias("daily_orders"),
    spark_round(spark_sum("units_sold"),      2).alias("daily_units"),
    spark_round(spark_sum("sales_amount"),    2).alias("daily_sales"),
    spark_round(spark_sum("profit_per_order"),2).alias("daily_profit"),
    spark_round(avg("delay_days"),            2).alias("avg_daily_delay")
)

# 4. Demand forecast
from pyspark.sql.functions import date_trunc
from pyspark.sql.window import Window

df_weekly = df \
    .withColumn("week", date_trunc("week", col("order_date"))) \
    .groupBy("warehouse_id", "category", "week") \
    .agg(
        spark_round(spark_sum("units_sold"),  2).alias("weekly_units"),
        spark_round(spark_sum("sales_amount"),2).alias("weekly_sales")
    )

window_4w = Window \
    .partitionBy("warehouse_id", "category") \
    .orderBy("week") \
    .rowsBetween(-3, 0)

df_demand_forecast = df_weekly \
    .withColumn(
        "moving_avg_4w_units",
        spark_round(avg("weekly_units").over(window_4w), 1)
    ) \
    .withColumn(
        "moving_avg_4w_sales",
        spark_round(avg("weekly_sales").over(window_4w), 2)
    )

# 5. Moving average delay
df_daily_delay = df.groupBy("route", "order_date").agg(
    spark_round(avg("delay_days"), 2).alias("avg_daily_delay"),
    count("*").alias("daily_orders")
)

window_7d = Window \
    .partitionBy("route") \
    .orderBy("order_date") \
    .rowsBetween(-6, 0)

df_moving_avg = df_daily_delay \
    .withColumn(
        "moving_avg_7d_delay",
        spark_round(avg("avg_daily_delay").over(window_7d), 2)
    )

# ── Export all tables ──
pdf_route     = export_table(df_route_delay,     "delay_summary")
pdf_warehouse = export_table(df_warehouse,        "warehouse_summary")
pdf_daily_wh  = export_table(df_daily_warehouse,  "daily_warehouse_metrics")
pdf_forecast  = export_table(df_demand_forecast,  "demand_forecast")
pdf_moving    = export_table(df_moving_avg,       "moving_avg_delay")

print("\n  All tables exported successfully.")


  EXPORTING ALL TABLES → SQL + CSV
  [SAVED] delay_summary                  →     92 rows
  [SAVED] warehouse_summary              →     38 rows
  [SAVED] daily_warehouse_metrics        →  7,918 rows
  [SAVED] demand_forecast                →  3,541 rows
  [SAVED] moving_avg_delay               →  4,640 rows

  All tables exported successfully.


## -------------Data Validation---------------

## 1. Row Count Check

In [3]:
print("\n" + "="*50)
print("  VALIDATION 1: Row Count Integrity")
print("="*50)

# Raw original file row count
df_raw = spark.read.csv(
    "DataCoSupplyChainDataset.csv",
    header=True,
    inferSchema=True,
    encoding="ISO-8859-1"
)
raw_count       = df_raw.count()
cleaned_count   = df.count()
lost            = raw_count - cleaned_count

print(f"  Raw file rows       : {raw_count:,}")
print(f"  After cleaning      : {cleaned_count:,}")
print(f"  Rows removed        : {lost:,}  (duplicates + nulls)")

if lost / raw_count * 100 < 5:
    print("  [PASS] Less than 5% data removed — acceptable")
else:
    print(f"  [WARN] {lost/raw_count*100:.1f}% data removed — review cleaning steps")


  VALIDATION 1: Row Count Integrity
  Raw file rows       : 180,519
  After cleaning      : 180,519
  Rows removed        : 0  (duplicates + nulls)
  [PASS] Less than 5% data removed — acceptable


## 2. Null Check on Key Columns

In [4]:
print("\n" + "="*50)
print("  VALIDATION 2: Null Check on Key Columns")
print("="*50)

key_columns = [
    "order_id", "route", "warehouse_id",
    "eta_days", "ata_days", "delay_days",
    "units_sold", "sales_amount", "order_date"
]

all_passed = True
for c in key_columns:
    null_count = df.filter(col(c).isNull()).count()
    status = "[PASS]" if null_count == 0 else "[FAIL]"
    if null_count > 0:
        all_passed = False
    print(f"  {status} {c:25s}: {null_count:,} nulls")

if all_passed:
    print("\n  All key columns are null-free.")
else:
    print("\n  [WARN] Some nulls remain — check cleaning step.")


  VALIDATION 2: Null Check on Key Columns
  [PASS] order_id                 : 0 nulls
  [PASS] route                    : 0 nulls
  [PASS] warehouse_id             : 0 nulls
  [PASS] eta_days                 : 0 nulls
  [PASS] ata_days                 : 0 nulls
  [PASS] delay_days               : 0 nulls
  [PASS] units_sold               : 0 nulls
  [PASS] sales_amount             : 0 nulls
  [PASS] order_date               : 0 nulls

  All key columns are null-free.


## 3. Aggregation Cross-Check

In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("LogiScale_Phase3") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [7]:
from pyspark.sql.functions import avg, col

print("\n" + "="*50)
print("  VALIDATION 3: Aggregation Cross-Check (FIXED)")
print("="*50)

# Step 1: Pick one route
route_name = df.select("route").distinct().limit(1).collect()[0][0]

# Step 2: PySpark calculation (RAW DATA)
spark_avg_val = df.filter(col("route") == route_name) \
    .agg(avg("delay_days").alias("avg_delay")) \
    .collect()[0]["avg_delay"]

spark_avg_val = round(spark_avg_val, 2)

# Step 3: Pandas calculation (same RAW DATA)
pdf_check = df.filter(col("route") == route_name) \
    .select("delay_days") \
    .toPandas()

pandas_avg_val = round(pdf_check["delay_days"].mean(), 2)

# Step 4: Compare
difference = abs(spark_avg_val - pandas_avg_val)

print(f"\n  Route tested            : {route_name}")
print(f"  PySpark avg delay       : {spark_avg_val}")
print(f"  Pandas  avg delay       : {pandas_avg_val}")
print(f"  Difference              : {difference}")

if difference <= 0.01:
    print("  [PASS] Aggregation is consistent")
else:
    print("  [FAIL] Still mismatch — investigate data types")


  VALIDATION 3: Aggregation Cross-Check (FIXED)

  Route tested            : South America
  PySpark avg delay       : 0.56
  Pandas  avg delay       : 0.56
  Difference              : 0.0
  [PASS] Aggregation is consistent
